# Modelagem Preditiva — Employee Attrition

Com os dados devidamente explorados e tratados na etapa de EDA, partimos agora para a **modelagem preditiva**. O objetivo é treinar e avaliar modelos de classificação utilizando o `scikit-learn` para prever se um funcionário irá deixar a empresa (`Attrition = Yes`) ou permanecer (`Attrition = No`).

## Importações

Vou realizar aqui uma importação completa das principais ferramentas que serão utilizadas. Conforme cada componente for sendo aplicado ao longo da modelagem, será feita uma explicação sobre a função e sobre como ele é utilizado dentro do fluxo de trabalho do machine learning com o scikit-learn.

In [120]:
import pandas as pd
import numpy as np
import seaborn as sns


from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder, OneHotEncoder, PowerTransformer, StandardScaler
#lineares
from sklearn.linear_model import LogisticRegression

#árvores
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

#svm
from sklearn.svm import SVC

#kNN
from sklearn.neighbors import KNeighborsClassifier

RAMDOM_STATE = 42   

from src.config import DADOS_TRATADOS

In [121]:
df = pd.read_parquet(DADOS_TRATADOS)

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


In [122]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EnvironmentSatisfaction   1470 non-null   int64
 9   Gender                    1470 non-null   str  
 10  HourlyRate                1470 non-null   int64
 11  JobInvolvement            1470 non-null   int64
 12  JobLevel                  1470 non-null   int64
 13  JobRole                   1470 non-null   str  
 14  JobSatisfaction           1470 non-null   int64
 15

## Definições de colunas e preprocessamento

### Tipos da colunas

Com base na leitura da base devidimos as colunas nos seguintes tipos

In [123]:
colunas_categoricas_nao_ordenadas = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime",
]

colunas_categoricas_ordenadas = [
    "Education",
    "EnvironmentSatisfaction",
    "JobSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance",
]

coluna_alvo = ["Attrition"]


colunas_numericas = [
    coluna for coluna in df.columns if coluna not in (
        colunas_categoricas_nao_ordenadas + colunas_categoricas_ordenadas + coluna_alvo
    )
]

In [124]:
colunas_numericas_min_max = ["DailyRate", "HourlyRate", "MonthlyRate"]
colunas_numericas_std = ["Age"]
colunas_numericas_power_transform = [coluna for coluna in colunas_numericas if coluna not in (colunas_numericas_min_max + colunas_numericas_std)]

In [125]:
X = df.drop(columns='Attrition')

y = df['Attrition']

y[:20]

0     Yes
1      No
2     Yes
3      No
4      No
5      No
6      No
7      No
8      No
9      No
10     No
11     No
12     No
13     No
14    Yes
15     No
16     No
17     No
18     No
19     No
Name: Attrition, dtype: str

In [126]:
le = LabelEncoder()

y = le.fit_transform(y)

y[:20]

array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [127]:
le.classes_

array(['No', 'Yes'], dtype=object)

### Separando a base em treino e teste

In [128]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RAMDOM_STATE, stratify=y)

### Preprocessamento das colunas

In [129]:
preprocessamento_arvore = ColumnTransformer([
    ("one_hot", OneHotEncoder(drop='first'), colunas_categoricas_nao_ordenadas),
    ("ordinal", OrdinalEncoder(categories='auto'), colunas_categoricas_ordenadas),
])

preprocessamento = ColumnTransformer([
    ("one_hot", OneHotEncoder(drop='first'), colunas_categoricas_nao_ordenadas),
    ("ordinal", OrdinalEncoder(categories='auto'), colunas_categoricas_ordenadas),
    ("min_max", MinMaxScaler(), colunas_numericas_min_max),
    ("stdscaler", StandardScaler(), colunas_numericas_std),
    ("power_transform", PowerTransformer(), colunas_numericas_power_transform)
])


O pré-processamento dos dados foi realizado utilizando o `ColumnTransformer`, permitindo aplicar transformações diferentes para grupos específicos de colunas. As variáveis categóricas não ordenadas foram tratadas com `OneHotEncoder`, que transforma cada categoria em variáveis binárias, enquanto as categóricas ordenadas foram codificadas com `OrdinalEncoder`, preservando a relação de ordem existente entre os níveis da variável. No caso das variáveis numéricas, foram aplicadas diferentes transformações dependendo do comportamento observado nas distribuições dos dados. O `MinMaxScaler` foi utilizado para reescalar variáveis dentro de um intervalo definido quando sua distribuição já apresentava comportamento relativamente estável; o `StandardScaler` foi aplicado em variáveis com distribuição aproximadamente normal, realizando padronização para média zero e desvio padrão um; e o `PowerTransformer` foi empregado em variáveis com assimetria, com o objetivo de aproximar a distribuição de uma forma mais próxima da normalidade. 

Para modelos baseados em árvores, foi utilizado um pré-processamento mais simples, restringindo as transformações às variáveis categóricas, já que esse tipo de modelo não é sensível à escala das variáveis numéricas. É importante destacar que essas escolhas não foram feitas de forma arbitrária, mas sim baseadas na análise exploratória dos dados, especialmente na observação de gráficos de distribuição e outras características estatísticas das variáveis, além da consideração do tipo de cada coluna.

## Classificadores no treino

### Definindo classificadores

In [130]:
scale_pos_weight = np.bincount(y)[0] / np.bincount(y)[1]

In [131]:
classificadores = {
    "LogisticRegression": {
        "preprocessor": preprocessamento,
        "classificador": LogisticRegression(class_weight='balanced'),
    },
    "DecisionTreeClassifier": {
        "preprocessor": preprocessamento_arvore,
        "classificador": DecisionTreeClassifier(class_weight='balanced'),
    },
    "RandomForestClassifier": {
        "preprocessor": preprocessamento_arvore,
        "classificador": RandomForestClassifier(class_weight='balanced'),
    },
    "LGBMClassifier": {
        "preprocessor": preprocessamento_arvore,
        "classificador": LGBMClassifier(random_state=RAMDOM_STATE, verbose=-1, scale_pos_weight=scale_pos_weight),
    },
    "XGBClassifier": {
        "preprocessor": preprocessamento_arvore,
        "classificador": XGBClassifier(random_state=RAMDOM_STATE, scale_pos_weight=scale_pos_weight),
    },
    "SVC": {
        "preprocessor": preprocessamento,
        "classificador": SVC(class_weight='balanced'),
    },    
    "KNeighborsClassifier": {
        "preprocessor": preprocessamento,
        "classificador": KNeighborsClassifier(),
    },
}

### Definindo Pipeline

In [132]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RAMDOM_STATE)

O `StratifiedGroupKFold` é um método de **validação cruzada** que divide os dados em vários subconjuntos (folds) mantendo duas propriedades importantes: a **proporção das classes semelhante em cada divisão** (estratificação) e o **agrupamento de amostras pertencentes ao mesmo grupo** (evitando que dados do mesmo grupo apareçam ao mesmo tempo no treino e na validação).

No `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)`, o conjunto de dados é dividido em **5 folds**, os dados são **embaralhados antes da divisão**, e `random_state` garante que a divisão seja **reprodutível**. Durante o treinamento, o modelo é treinado várias vezes: em cada iteração, quatro folds são usados para treino e um para validação.

Mesmo utilizando `train_test_split`, o `StratifiedGroupKFold` ainda é útil porque os dois têm papéis diferentes. O `train_test_split` cria um **conjunto de teste final** separado (por exemplo, 20% dos dados) que será usado apenas para avaliar o desempenho final do modelo. Já o `StratifiedGroupKFold` é aplicado **apenas no conjunto de treino** para fazer validação cruzada, permitindo estimar melhor a performance do modelo e reduzir o risco de overfitting durante o treinamento ou ajuste de hiperparâmetros.


In [133]:
scoring = [
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "recall",
    "roc_auc",
    "average_precision",
]

resultados = {}

for nome, config in classificadores.items():

    pipe = Pipeline([
        ("preprocessamento", config["preprocessor"]),
        ("pca", PCA(n_components=0.95, random_state=RAMDOM_STATE)),
        ("modelo", config["classificador"]),
    ])

    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

    medias = {metric: np.mean(scores[f"test_{metric}"]) for metric in scoring}

    resultados[nome] = medias

In [134]:
resultados = pd.DataFrame(resultados)
resultados

,LogisticRegression,DecisionTreeClassifier,RandomForestClassifier,LGBMClassifier,XGBClassifier,SVC,KNeighborsClassifier
accuracy,0.769542,0.788265,0.851190,0.838428,0.839268,0.830772,0.849488
balanced_accuracy,0.766958,0.584794,0.547970,0.640202,0.630090,0.733374,0.578821
f1,0.517219,0.296527,0.172320,0.402151,0.389593,0.529972,0.271877
precision,0.392164,0.316485,0.777778,0.509900,0.510132,0.485376,0.581905
recall,0.763158,0.284211,0.100000,0.347368,0.321053,0.589474,0.178947
roc_auc,0.828821,0.584794,0.765541,0.761547,0.752252,0.805436,0.702534
average_precision,0.628890,0.209427,0.467745,0.470497,0.449260,0.604068,0.360302


A análise comparativa dos modelos evidencia diferenças relevantes de desempenho, especialmente quando se observam métricas além da acurácia global, como *balanced accuracy*, *F1-score*, *recall* e *ROC AUC*, que são mais adequadas em cenários com possível desbalanceamento de classes.

O **KNeighborsClassifier** apresentou a maior acurácia (≈0,85), seguido de perto pelo **RandomForestClassifier** e pelo **XGBClassifier**, indicando forte capacidade preditiva geral. No entanto, ao observar métricas mais sensíveis ao equilíbrio das classes, nota-se que o desempenho do KNN não se mantém proporcionalmente, sugerindo possível viés em relação à classe majoritária.

O **RandomForestClassifier** se destaca como o modelo mais equilibrado no conjunto de métricas. Apesar de um *recall* relativamente baixo, ele mantém bons níveis de acurácia, *ROC AUC* e *average precision*, indicando robustez geral e boa capacidade discriminativa. Já o **XGBClassifier** e o **LGBMClassifier** apresentam desempenho competitivo e consistente, sendo alternativas eficientes, especialmente em contextos onde há necessidade de modelos escaláveis e com bom trade-off entre viés e variância.

O **LogisticRegression** demonstra um comportamento mais estável em termos de *recall* (≈0,76), o que indica maior capacidade de identificar a classe positiva, embora com penalização em precisão. Isso sugere um perfil mais conservador, potencialmente útil em aplicações onde falsos negativos são mais críticos.

Por outro lado, o **DecisionTreeClassifier** apresenta desempenho inferior na maioria das métricas, especialmente em *F1-score* e *ROC AUC*, o que pode indicar sobreajuste ou baixa capacidade de generalização isoladamente. Já o **SVC** apresenta um equilíbrio intermediário, com bons valores de *ROC AUC* e *recall*, mas sem se destacar significativamente em nenhuma métrica específica.


Portanto, mesmo sendo um modelo linear, a regressão logística demonstrou o comportamento mais equilibrado entre as métricas avaliadas. Esse resultado não é incomum, especialmente em cenários onde a separação entre as classes pode ser bem aproximada por uma fronteira linear.

É importante destacar que o fato de ser um modelo linear não implica desempenho inferior. Pelo contrário, modelos lineares como a regressão logística são amplamente reconhecidos por sua robustez, interpretabilidade e capacidade de generalização, sobretudo em contextos com menor complexidade estrutural ou com bom pré-processamento dos dados.

Além disso, sua estabilidade em métricas como *recall* e *balanced accuracy* sugere uma melhor adaptação a possíveis desbalanceamentos de classe, o que reforça sua adequação para problemas onde o custo de erro não é uniforme entre as classes.

Em síntese, a regressão logística se apresenta como uma solução adequada para a base em análise. Diante desse desempenho consistente e equilibrado entre as métricas, ela será adotada como modelo de referência para as próximas etapas do projeto.

A partir deste ponto, o foco será o refinamento do modelo por meio de *tuning* de hiperparâmetros, utilizando técnicas como *Grid Search*, com o objetivo de otimizar seu desempenho e explorar possíveis ganhos adicionais de generalização.



In [135]:
config = classificadores["LogisticRegression"]

pipe = Pipeline([
    ("preprocessamento", config["preprocessor"]),
    ("pca", PCA(n_components=0.95, random_state=RAMDOM_STATE)),
    ("modelo", config["classificador"]),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [136]:
indexs = []
columns = []
for classe in le.classes_:
    indexs.append(f'Eram {classe}')
    columns.append(f'O modelo preveu {classe}')

cm = confusion_matrix(y_test, y_pred)

tabela = pd.DataFrame(cm,index=indexs,columns=columns)

tabela

,O modelo preveu No,O modelo preveu Yes
Eram No,193,54
Eram Yes,16,31


## GridSearch